In [1]:
import pandas as pd
import numpy as np
import pymysql
import os

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix


In [208]:
# timeout = 10
# db_config = {
#     'charset': "utf8mb4",
#     'connect_timeout': timeout,   
#     'cursorclass': pymysql.cursors.DictCursor,
#     'db': os.getenv('DB_NAME'),
#     'host': os.getenv('DB_HOST'),
#     'password': os.getenv('DB_PASSWORD'),
#     'port': int(os.getenv('DB_PORT')),
#     'user': os.getenv('DB_USER'),
#     'write_timeout': timeout,
#     'read_timeout': timeout
# }

In [209]:
# conn = pymysql.connect(**db_config)
# cursor = conn.cursor()
# print("✅ Connected to MySQL database.")

# query = "SELECT * FROM market_data"  
# cursor.execute(query)

# # Load data into pandas DataFrame
# df = pd.DataFrame(cursor.fetchall())

# # Close the cursor and connection
# cursor.close()
# conn.close()

In [210]:
# # Ticker auswählren
# df=df[df.ticker=='SXR8.DE']
# df.head()

In [11]:
import yfinance as yf
interval='15m'
period='60d'
df = yf.download('SXR8.DE', interval=interval, period=period, progress=False).reset_index()
df.columns = [col[0] for col in df.columns] 
df.columns = ['timestamp', 'close', 'high', 'low', 'open', 'volume']

In [12]:
df.head()

,timestamp,close,high,low,open,volume
0,2025-02-06 08:00:00+00:00,621.359985,621.520020,620.539978,620.539978,592
1,2025-02-06 08:15:00+00:00,622.219971,622.219971,621.119995,621.419983,670
2,2025-02-06 08:30:00+00:00,622.919983,622.940002,622.059998,622.179993,564
3,2025-02-06 08:45:00+00:00,622.539978,623.119995,622.419983,622.840027,2059
4,2025-02-06 09:00:00+00:00,622.659973,622.799988,622.119995,622.419983,1539


📈 1. Lag Features (most critical)

In [213]:
# 15, 30, 45, 60, 90 minutes behind
lags = [1, 2, 3, 4, 6]
for lag in lags:
    df[f'lag_close_{lag}'] = df['close'].shift(lag)
    df[f'lag_volume_{lag}'] = df['volume'].shift(lag)


🔄 2. Rolling Window Features (trend + volatility)

In [214]:
# e.g., 3 = 45 mins, 6 = 90 mins, 12 = 3 hours
windows = [3, 6, 12]

for w in windows:
    df[f'rolling_mean_close_{w}'] = df['close'].rolling(window=w).mean()
    df[f'rolling_std_close_{w}'] = df['close'].rolling(window=w).std()
    df[f'rolling_max_close_{w}'] = df['close'].rolling(window=w).max()
    df[f'rolling_min_close_{w}'] = df['close'].rolling(window=w).min()
    df[f'rolling_mean_volume_{w}'] = df['volume'].rolling(window=w).mean()


🔥 3. Price Action Features (momentum & volatility)


In [215]:
df['return_15min'] = df['close'].pct_change()
df['return_30min'] = df['close'].pct_change(2)
df['return_1h'] = df['close'].pct_change(4)
df['high_low_spread'] = df['high'] - df['low']
df['candle_body'] = abs(df['close'] - df['open'])
df['upper_shadow'] = df['high'] - df[['open', 'close']].max(axis=1)
df['lower_shadow'] = df[['open', 'close']].min(axis=1) - df['low']


🧭 4. Time Features (captures patterns)

In [216]:
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek
df['is_opening'] = (df['hour'] == 9).astype(int)
df['is_closing'] = (df['hour'] == 16).astype(int)


⚙️ 5. Technical Indicators (TA features)

In [217]:
from ta.momentum import RSIIndicator
from ta.trend import MACD, EMAIndicator

df['rsi_14'] = RSIIndicator(close=df['close'], window=14).rsi()
df['ema_9'] = EMAIndicator(close=df['close'], window=9).ema_indicator()
df['ema_21'] = EMAIndicator(close=df['close'], window=21).ema_indicator()

macd = MACD(close=df['close'])
df['macd'] = macd.macd()
df['macd_signal'] = macd.macd_signal()
df['macd_diff'] = macd.macd_diff()


6. Target Variable

In [218]:
# # Regression: Predict next 15-min close
# df['target'] = df['close'].shift(-1)

# Classification: Predict if next close is higher
df['target_up'] = (df['close'].shift(-1) > df['close']).astype(int)


📋 7. Final Cleanup

In [219]:
df = df.sort_values('timestamp')

# Set index if you prefer time-based ops
df.set_index('timestamp', inplace=True)

# Drop rows with NaN from rolling/lags or target
df.dropna(inplace=True)

# Define the raw price columns we want to exclude
raw_price_columns = ['open', 'high', 'low', 'close', 'volume']

# Include only features (exclude raw prices and target)
exclude = raw_price_columns + ['target_up']  # Exclude raw price columns and target column
features = [col for col in df.columns if col not in exclude]

X = df[features]
y = df['target_up']

🧪 8. Train/Test Split


In [220]:
# Time-aware split: don't shuffle for time series
split_index = int(0.8 * len(df))
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]


9. 🤖 Train XGBoost Classifier

In [221]:
model = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss'
)

model.fit(X_train, y_train)


c:\Users\andre\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [22:56:39] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)

10. 📊 Evaluate Model


In [222]:
y_pred = model.predict(X_test)

print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, digits=4))


Confusion Matrix:
 [[118  76]
 [119  88]]

Classification Report:
               precision    recall  f1-score   support

           0     0.4979    0.6082    0.5476       194
           1     0.5366    0.4251    0.4744       207

    accuracy                         0.5137       401
   macro avg     0.5172    0.5167    0.5110       401
weighted avg     0.5179    0.5137    0.5098       401



# ✅ How to Make Real-Time Predictions Using Your Model


🧭 Step 1: Collect Latest OHLCV Data

You need the most recent:

- Close

 - Open

 - High

- Low

- Volume

This is usually:

- The current bar if it's completed (e.g., last 15-min candle).

- Or the last full bar for lag-safe models.

🧠 Step 2: Build Feature Vector from That Data

For example, let’s say your data runs up to 2025-04-29 12:30, and you want to make a prediction at 12:45. You’ll:

1. Grab the last completed 15-min bar (e.g., 12:30)

2. Take the past N intervals (lags, rolling windows, indicators)

3. Build a single row DataFrame of features (same structure as training data)

In [263]:
def build_features_for_prediction(recent_df: pd.DataFrame) -> pd.DataFrame:
    """Takes recent OHLCV data and returns the latest feature row for prediction"""
    df = recent_df.copy()

    # Lag features
    lags = [1, 2, 3, 4, 6]
    for lag in lags:
        df[f'lag_close_{lag}'] = df['close'].shift(lag)
        df[f'lag_volume_{lag}'] = df['volume'].shift(lag)

    # Rolling features
    windows = [3, 6, 12]
    for w in windows:
        df[f'rolling_mean_close_{w}'] = df['close'].rolling(window=w).mean()
        df[f'rolling_std_close_{w}'] = df['close'].rolling(window=w).std()
        df[f'rolling_max_close_{w}'] = df['close'].rolling(window=w).max()
        df[f'rolling_min_close_{w}'] = df['close'].rolling(window=w).min()
        df[f'rolling_mean_volume_{w}'] = df['volume'].rolling(window=w).mean()

    # Add the exact same features as in the training pipeline
    df['return_15min'] = df['close'].pct_change()
    df['return_30min'] = df['close'].pct_change(2)
    df['return_1h'] = df['close'].pct_change(4)
    
    df['high_low_spread'] = df['high'] - df['low']
    df['candle_body'] = abs(df['close'] - df['open'])
    df['upper_shadow'] = df['high'] - df[['open', 'close']].max(axis=1)
    df['lower_shadow'] = df[['open', 'close']].min(axis=1) - df['low']

    df['hour'] = df['timestamp'].dt.hour
    df['day_of_week'] = df['timestamp'].dt.dayofweek
    df['is_opening'] = (df['hour'] == 9).astype(int)
    df['is_closing'] = (df['hour'] == 16).astype(int)

    df['rsi_14'] = RSIIndicator(close=df['close'], window=14).rsi()
    df['ema_9'] = EMAIndicator(close=df['close'], window=9).ema_indicator()
    df['ema_21'] = EMAIndicator(close=df['close'], window=21).ema_indicator()

    macd = MACD(close=df['close'])
    df['macd'] = macd.macd()
    df['macd_signal'] = macd.macd_signal()
    df['macd_diff'] = macd.macd_diff()  

    df = df.sort_values('timestamp')

    # Set index if you prefer time-based ops
    df.set_index('timestamp', inplace=True)

    # Drop NAs from recent window calc
    df.dropna(inplace=True)

    # Define the raw price columns we want to exclude
    raw_price_columns = ['open', 'high', 'low', 'close', 'volume']

    # Include only features (exclude raw prices and target)
    exclude = raw_price_columns + ['target_up']  # Exclude raw price columns and target column
    features = [col for col in df.columns if col not in exclude]

    # Return the last row only (most recent complete feature set)
    return df[features].iloc[[-1]]


🚀 Step 3: Use the Model to Predict

In [282]:
interval='15m'
period='1d'
df_recent = yf.download('SXR8.DE', interval=interval, period=period, progress=False).reset_index()
df_recent.columns = [col[0] for col in df_recent.columns] 
df_recent.columns = ['timestamp', 'close', 'high', 'low', 'open', 'volume']

In [285]:
df_recent.head()

,timestamp,close,high,low,open,volume
0,2025-04-29 07:00:00+00:00,517.039978,517.359985,516.460022,516.500000,1789
1,2025-04-29 07:15:00+00:00,517.739990,517.919983,517.080017,517.080017,572
2,2025-04-29 07:30:00+00:00,517.200012,517.799988,517.119995,517.719971,456
3,2025-04-29 07:45:00+00:00,517.640015,517.820007,517.059998,517.080017,254
4,2025-04-29 08:00:00+00:00,516.940002,517.599976,516.820007,517.500000,403


In [284]:
# Suppose `model` is your trained XGBoost model
latest_features = build_features_for_prediction(df_recent)
prediction = model.predict(latest_features)[0]

if prediction == 1:
    print("Model predicts: 📈 Price will go UP")
else:
    print("Model predicts: 📉 Price will go DOWN")


Model predicts: 📉 Price will go DOWN
